# KDSC Extreme-Sample GIFs (AV2 + Waymo)

Visualize sample agents from each KDSC cluster (the IROS-2025-paper baseline) as GIFs with a speed / relative-speed / long-accel / lat-accel dashboard.

- Loads labels produced by `kdsc_replication.ipynb` (`artifacts/kdsc_replication/<dataset>/{dataset}_val_kdsc.parquet`).
- Uses the same custom scene renderer as the Waymo GIF notebook (only the center agent is red) so the same code path works for both AV2 and Waymo.
- KDSC clustering uses agglomerative + standardized centroids, so 'extremeness' is computed against the saved `centroids_standardized` (not a KMeans model).

Outputs go to `artifacts/kdsc_replication/<dataset>/figures/gifs/`.

In [1]:
from __future__ import annotations
import sys, json, pickle
from pathlib import Path

import numpy as np
import pandas as pd

# -------- DATASET SWITCH --------
DATASET = 'av2'         # 'av2' or 'waymo'
SPLIT   = 'val'         # waymo split; for av2 the scenario_viz uses 'val' too
# --------------------------------

_CWD = Path.cwd()
for cand in (_CWD, _CWD.parent, _CWD.parent.parent):
    if (cand / 'src').is_dir():
        REPO_ROOT = cand
        break
else:
    REPO_ROOT = _CWD
SRC_ROOT = REPO_ROOT / 'src'
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from tailrisk_mp.runtime import ensure_numpy_pickle_compat
ensure_numpy_pickle_compat()

KDSC_DIR = REPO_ROOT / 'artifacts' / 'kdsc_replication' / DATASET
GIF_DIR  = KDSC_DIR / 'figures' / 'gifs'
GIF_DIR.mkdir(parents=True, exist_ok=True)

VAL_PARQUET = KDSC_DIR / f'{DATASET}_val_kdsc.parquet'
MODEL_PKL   = KDSC_DIR / 'kdsc_agglomerative_K2.pkl'

for p in (VAL_PARQUET, MODEL_PKL):
    if not p.exists():
        raise FileNotFoundError(f'Missing {p} - run kdsc_replication.ipynb (DATASET={DATASET!r}) first.')

val = pd.read_parquet(VAL_PARQUET)
with open(MODEL_PKL, 'rb') as f:
    bundle = pickle.load(f)
scaler    = bundle['scaler']
centroids = np.asarray(bundle['centroids_standardized'])  # (2, n_features) in standardized space
FEATURES  = bundle['features']
LABEL_MAP = bundle['label_map']

print('dataset      :', DATASET)
print('val rows     :', len(val))
print('cluster sizes:'); print(val['style_label_kdsc'].value_counts())
print('label map    :', LABEL_MAP)
print('features     :', FEATURES)

dataset      : av2
val rows     : 21809
cluster sizes:
style_label_kdsc
normal        21725
aggressive       84
Name: count, dtype: int64
label map    : {1: 'aggressive', 0: 'normal'}
features     : ['max_abs_accel', 'var_acceleration', 'var_speed', 'gamma']


## 1. Score 'extremeness' inside each cluster

Same definition as the main pipeline: distance to the *other* centroid minus distance to the *own* centroid, in standardized feature space. Higher = more separated from the opposing cluster.

In [2]:
X = val[FEATURES].astype(float).to_numpy()
Xs = scaler.transform(X)

d0 = np.linalg.norm(Xs - centroids[0], axis=1)
d1 = np.linalg.norm(Xs - centroids[1], axis=1)

val = val.copy()
val['dist_to_other_center'] = np.where(val['cluster_id_K2'] == 0, d1, d0)
val['dist_to_own_center']   = np.where(val['cluster_id_K2'] == 0, d0, d1)
val['extremeness'] = val['dist_to_other_center'] - val['dist_to_own_center']
val[['style_label_kdsc', 'cluster_id_K2', 'extremeness']].groupby('style_label_kdsc').describe().round(2)

cluster_id_K2                                    extremeness  \
                         count mean  std  min  25%  50%  75%  max       count   
style_label_kdsc                                                                
aggressive                84.0  1.0  0.0  1.0  1.0  1.0  1.0  1.0        84.0   
normal                 21725.0  0.0  0.0  0.0  0.0  0.0  0.0  0.0     21725.0   

                                                                 
                   mean   std   min    25%    50%    75%    max  
style_label_kdsc                                                 
aggressive        13.48  6.17  0.08   7.88  15.90  19.22  20.38  
normal            19.65  1.77  0.06  19.71  20.33  20.43  20.64

## 2. Pick samples per cluster

Random sampling per cluster (same choice as our main GIF notebooks). Flip to `nlargest('extremeness')` if you want the most-separated samples instead.

In [3]:
PER_CLUSTER = 5
RANDOM_STATE = 42

picks: dict[str, pd.DataFrame] = {}
for label in val['style_label_kdsc'].dropna().unique():
    sub = val[val['style_label_kdsc'] == label].dropna(subset=['scenario_id', 'center_objects_id'])
    top = sub.sample(min(PER_CLUSTER, len(sub)), random_state=RANDOM_STATE)
    # top = sub.nlargest(PER_CLUSTER, 'extremeness')
    picks[label] = top
    print(f'\n=== {label} ({len(top)} samples) ===')
    cols = ['scenario_id', 'center_objects_id', 'extremeness',
            'avg_speed', 'max_abs_accel', 'max_abs_jerk',
            'hard_brake_count', 'hard_accel_count']
    cols = [c for c in cols if c in top.columns]
    print(top[cols].round(3).to_string(index=False))


=== normal (5 samples) ===
                         scenario_id  center_objects_id  extremeness  avg_speed  max_abs_accel  max_abs_jerk  hard_brake_count  hard_accel_count
9f8e22b9-2eae-4de1-8816-3517b3101a71              43633       20.554      4.062          2.198        11.214                 0                 0
3b9b45c5-023f-40e8-83de-0743cc8faee7              59219       20.470      9.201          1.373         5.385                 0                 0
a912275c-830e-4e0f-b7bf-ee461d60d269             154405       20.373     10.136          2.718        21.947                 0                 0
d974b9f7-9ed2-4671-adcd-0c54a711665b              36162       18.724      4.513         10.457        57.708                 7                 7
06f616af-3a2e-4607-8890-20b3cc976e6f              51659       20.514      7.980          2.085         7.016                 0                 0

=== aggressive (5 samples) ===
                         scenario_id  center_objects_id  extremeness  

## 3. Renderer (custom: only center red)

Identical to the Waymo GIF renderer. Works for both AV2 and Waymo because every non-center agent is colored by track-type only, never by `tracks_to_predict` membership.

In [4]:
import io
import matplotlib.pyplot as plt
from tailrisk_mp.scenario_viz import (
    load_scenario_by_id, _normalize_track_id, _map_style, _track_color,
    _valid_segments, _scenario_center,
)

DT  = 0.1
G   = 9.81
KMH = 3.6

SPEED_SMOOTH_FRAMES = 9
ACCEL_SMOOTH_FRAMES = 1

V_MAX_KMH       = 140.0
SPEED_WARN_KMH  =  60.0
SPEED_LIMIT_KMH =  90.0

V_REL_MAX_KMH    = 50.0
REL_WARN_KMH     = 15.0
REL_LIMIT_KMH    = 30.0

NEIGHBOR_RADIUS_M    = 30.0
NEIGHBOR_LAT_M       =  4.0
NEIGHBOR_DIR_DEG     = 30.0

A_LONG_MAX_G    = 3.0
ACC_LONG_WARN_G = 0.5
ACC_LONG_LIMIT_G= 0.8

A_LAT_MAX_G     = 1.5
ACC_LAT_WARN_G  = 0.3
ACC_LAT_LIMIT_G = 0.6

GREEN, ORANGE, RED = '#2ca02c', '#ff7f0e', '#d62728'

def _draw_scene_frame(ax, scenario, frame, center_track_id, view_radius):
    ax.clear()
    center_track_norm = _normalize_track_id(center_track_id)

    for feature in scenario['map_features'].values():
        poly = np.asarray(feature.get('polyline', []))
        if poly.ndim != 2 or len(poly) < 2:
            continue
        style = _map_style(feature.get('type', 'UNKNOWN'))
        ax.plot(poly[:, 0], poly[:, 1], color=style['color'],
                linewidth=style['linewidth'], zorder=1)

    tracks = scenario['tracks']
    center_xy = _scenario_center(scenario)
    for tid, t in tracks.items():
        if _normalize_track_id(tid) != center_track_norm and str(tid) != str(center_track_id):
            continue
        pos = np.asarray(t['state']['position'])
        valid = np.asarray(t['state']['valid']).reshape(-1).astype(bool)
        if frame < len(valid) and valid[frame]:
            center_xy = pos[frame, :2]
        break

    for track_id, track in tracks.items():
        state = track['state']
        positions = np.asarray(state['position'])
        valid = np.asarray(state['valid']).reshape(-1).astype(bool)
        norm_id = _normalize_track_id(track_id)
        is_center = norm_id == center_track_norm or str(track_id) == str(center_track_id)

        if is_center:
            color = RED
            line_width = 3.0
        else:
            color = _track_color(track.get('type', 'UNKNOWN'), 'other')
            line_width = 1.4

        history = valid.copy(); history[frame + 1:] = False
        future  = valid.copy(); future[: frame + 1] = False

        for segment in _valid_segments(positions, history):
            ax.plot(segment[:, 0], segment[:, 1], color=color, linewidth=line_width, zorder=3)
        for segment in _valid_segments(positions, future):
            ax.plot(segment[:, 0], segment[:, 1],
                    color=color, linewidth=max(1.0, line_width - 0.6),
                    linestyle='--', alpha=0.4, zorder=2)
        if frame < len(valid) and valid[frame]:
            xy = positions[frame, :2]
            ax.scatter(xy[0], xy[1], color=color,
                       s=90 if is_center else 18,
                       zorder=5, edgecolors='black' if is_center else 'none', linewidths=0.8)

    ax.set_aspect('equal')
    ax.set_xlim(center_xy[0] - view_radius, center_xy[0] + view_radius)
    ax.set_ylim(center_xy[1] - view_radius, center_xy[1] + view_radius)
    ax.axis('off')

def _zone_color(value, warn, red):
    a = abs(float(value))
    if a >= red:  return RED
    if a >= warn: return ORANGE
    return GREEN

def _smooth(arr, window):
    a = np.asarray(arr, dtype=float)
    if window <= 1 or a.size == 0:
        return a
    if a.ndim == 1:
        out = np.full_like(a, np.nan)
        half = window // 2
        T = len(a)
        for i in range(T):
            lo, hi = max(0, i - half), min(T, i + half + 1)
            seg = a[lo:hi]
            seg = seg[~np.isnan(seg)]
            if seg.size:
                out[i] = seg.mean()
        return out
    return np.stack([_smooth(a[:, j], window) for j in range(a.shape[1])], axis=1)

def _per_track_kinematics(positions, valid):
    T = len(valid)
    speed = np.full(T, np.nan)
    velocity = np.full((T, 2), np.nan)
    for i in range(1, T):
        if valid[i] and valid[i-1]:
            v = (positions[i] - positions[i-1]) / DT
            speed[i] = float(np.linalg.norm(v))
            velocity[i] = v
    return speed, velocity

def _compute_2d_accel(velocity):
    T = len(velocity)
    accel = np.full((T, 2), np.nan)
    for i in range(1, T):
        if not (np.any(np.isnan(velocity[i])) or np.any(np.isnan(velocity[i-1]))):
            accel[i] = (velocity[i] - velocity[i-1]) / DT
    return accel

def _decompose_long_lat(accel_vec, velocity):
    T = len(accel_vec)
    a_long = np.full(T, np.nan)
    a_lat  = np.full(T, np.nan)
    for i in range(T):
        if np.any(np.isnan(accel_vec[i])) or np.any(np.isnan(velocity[i])):
            continue
        spd = float(np.linalg.norm(velocity[i]))
        if spd < 0.1:
            continue
        h = velocity[i] / spd
        h_perp = np.array([-h[1], h[0]])
        a_long[i] = float(np.dot(accel_vec[i], h))
        a_lat[i]  = float(np.dot(accel_vec[i], h_perp))
    return a_long, a_lat

def _compute_relative_speed(scenario, ego_track, ego_pos, ego_valid, ego_speed, ego_vel):
    cos_thresh = float(np.cos(np.deg2rad(NEIGHBOR_DIR_DEG)))
    others = []
    for tid, t in scenario['tracks'].items():
        if t is ego_track:
            continue
        pos = np.asarray(t['state']['position'])[:, :2]
        v   = np.asarray(t['state']['valid']).reshape(-1).astype(bool)
        if pos.shape[0] < 2:
            continue
        s, vel = _per_track_kinematics(pos, v)
        s   = _smooth(s,   SPEED_SMOOTH_FRAMES)
        vel = _smooth(vel, SPEED_SMOOTH_FRAMES)
        others.append((pos, v, s, vel))

    T = len(ego_valid)
    rel = np.full(T, np.nan)
    for f in range(T):
        if not ego_valid[f] or np.isnan(ego_speed[f]):
            continue
        if ego_speed[f] < 1e-3 or np.any(np.isnan(ego_vel[f])):
            heading = np.array([1.0, 0.0])
        else:
            heading = ego_vel[f] / max(np.linalg.norm(ego_vel[f]), 1e-6)
        normal = np.array([-heading[1], heading[0]])

        neighbour_speeds: list[float] = []
        for pos, v, s, vel in others:
            if f >= len(v) or not v[f] or np.isnan(s[f]) or s[f] < 1e-3:
                continue
            d = pos[f] - ego_pos[f]
            if np.linalg.norm(d) > NEIGHBOR_RADIUS_M:
                continue
            if abs(float(np.dot(d, normal))) > NEIGHBOR_LAT_M:
                continue
            n_norm = max(np.linalg.norm(vel[f]), 1e-6)
            if float(np.dot(vel[f] / n_norm, heading)) < cos_thresh:
                continue
            neighbour_speeds.append(float(s[f]))

        rel[f] = (ego_speed[f] - float(np.mean(neighbour_speeds))) if neighbour_speeds else 0.0
    return rel

def _draw_speedometer(ax, speed_ms):
    ax.clear()
    speed_kmh = float(np.nan_to_num(speed_ms, nan=0.0)) * KMH
    theta = np.linspace(np.pi, 0, 200)
    speeds_at_theta = V_MAX_KMH * (np.pi - theta) / np.pi
    for i in range(len(theta) - 1):
        c = _zone_color(speeds_at_theta[i], SPEED_WARN_KMH, SPEED_LIMIT_KMH)
        ax.plot([np.cos(theta[i]), np.cos(theta[i+1])],
                [np.sin(theta[i]), np.sin(theta[i+1])],
                color=c, linewidth=10, solid_capstyle='butt')
    s_c = max(0.0, min(speed_kmh, V_MAX_KMH))
    angle = np.pi - np.pi * (s_c / V_MAX_KMH)
    ax.plot([0, 0.85 * np.cos(angle)], [0, 0.85 * np.sin(angle)],
            color='black', linewidth=3)
    ax.scatter([0], [0], s=40, color='black', zorder=5)
    color = _zone_color(speed_kmh, SPEED_WARN_KMH, SPEED_LIMIT_KMH)
    label = '--' if np.isnan(speed_ms) else f'{speed_kmh:.0f}'
    ax.text(0, -0.35, f'speed\n{label} km/h', ha='center', va='center',
            fontsize=10, color=color, fontweight='bold')
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-0.6, 1.2)
    ax.set_aspect('equal'); ax.axis('off')

def _draw_relative_speedometer(ax, rel_speed_ms):
    ax.clear()
    rel_kmh = float(np.nan_to_num(rel_speed_ms, nan=0.0)) * KMH
    theta = np.linspace(np.pi, 0, 200)
    rel_at_theta = -V_REL_MAX_KMH + 2 * V_REL_MAX_KMH * (np.pi - theta) / np.pi
    for i in range(len(theta) - 1):
        c = _zone_color(rel_at_theta[i], REL_WARN_KMH, REL_LIMIT_KMH)
        ax.plot([np.cos(theta[i]), np.cos(theta[i+1])],
                [np.sin(theta[i]), np.sin(theta[i+1])],
                color=c, linewidth=10, solid_capstyle='butt')
    r_c = max(-V_REL_MAX_KMH, min(rel_kmh, V_REL_MAX_KMH))
    angle = np.pi - np.pi * (r_c + V_REL_MAX_KMH) / (2 * V_REL_MAX_KMH)
    ax.plot([0, 0.85 * np.cos(angle)], [0, 0.85 * np.sin(angle)],
            color='black', linewidth=3)
    ax.scatter([0], [0], s=40, color='black', zorder=5)
    color = _zone_color(rel_kmh, REL_WARN_KMH, REL_LIMIT_KMH)
    label = '--' if np.isnan(rel_speed_ms) else f'{rel_kmh:+.0f}'
    ax.text(0, -0.35, f'rel speed\n{label} km/h', ha='center', va='center',
            fontsize=10, color=color, fontweight='bold')
    ax.set_xlim(-1.2, 1.2); ax.set_ylim(-0.6, 1.2)
    ax.set_aspect('equal'); ax.axis('off')

def _draw_accel_bar(ax, accel_ms2, *, label, amax_g, warn_g, limit_g,
                    neg_text='', pos_text=''):
    ax.clear()
    a_g = float(np.nan_to_num(accel_ms2, nan=0.0)) / G
    a_c = max(-amax_g, min(a_g, amax_g))
    xs = np.linspace(-amax_g, amax_g, 200)
    for i in range(len(xs) - 1):
        c = _zone_color(xs[i], warn_g, limit_g)
        ax.barh(0, xs[i+1] - xs[i], left=xs[i], color=c, alpha=0.22, height=0.6)
    color = _zone_color(a_g, warn_g, limit_g)
    ax.barh(0, a_c, left=0, color=color, height=0.6)
    ax.axvline(0, color='black', linewidth=1)
    val_txt = '--' if np.isnan(accel_ms2) else f'{a_g:+.2f}'
    ax.text(0, -0.55, f'{label}\n{val_txt} g', ha='center', va='center',
            fontsize=10, color=color, fontweight='bold')
    if neg_text:
        ax.text(-amax_g, 0.55, neg_text, ha='left', va='center', fontsize=8, color='#555')
    if pos_text:
        ax.text(amax_g, 0.55, pos_text, ha='right', va='center', fontsize=8, color='#555')
    ax.set_xlim(-amax_g * 1.05, amax_g * 1.05); ax.set_ylim(-1.0, 0.8)
    ax.axis('off')

def _resolve_track(scenario, center_objects_id):
    cid_norm = _normalize_track_id(center_objects_id)
    for tid, t in scenario['tracks'].items():
        if _normalize_track_id(tid) == cid_norm or str(tid) == str(center_objects_id):
            return t
    raise KeyError(f'center track {center_objects_id} not found')

def render_agent_gif_with_dashboard(*, dataset, scenario_id, center_objects_id, split,
                                    output_path, title='', view_radius=70.0,
                                    frame_stride=2, fps=10, max_frames=None):
    import imageio.v2 as imageio
    bundle = load_scenario_by_id(dataset=dataset, scenario_id=scenario_id, split=split)
    scenario = bundle['scenario']
    track = _resolve_track(scenario, center_objects_id)
    positions = np.asarray(track['state']['position'])[:, :2]
    valid = np.asarray(track['state']['valid']).reshape(-1).astype(bool)
    T = len(valid)

    raw_speed, raw_velocity = _per_track_kinematics(positions, valid)
    speed_per_frame_disp    = _smooth(raw_speed,    SPEED_SMOOTH_FRAMES)
    velocity_per_frame_disp = _smooth(raw_velocity, SPEED_SMOOTH_FRAMES)

    accel_vec = _compute_2d_accel(raw_velocity)
    a_long_raw, a_lat_raw = _decompose_long_lat(accel_vec, raw_velocity)
    a_long_per_frame = _smooth(a_long_raw, ACCEL_SMOOTH_FRAMES)
    a_lat_per_frame  = _smooth(a_lat_raw,  ACCEL_SMOOTH_FRAMES)

    rel_per_frame = _smooth(_compute_relative_speed(
        scenario, track, positions, valid,
        speed_per_frame_disp, velocity_per_frame_disp,
    ), SPEED_SMOOTH_FRAMES)

    frames = list(range(0, T, max(1, frame_stride)))
    if max_frames is not None:
        frames = frames[:max_frames]

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)

    fig = plt.figure(figsize=(12, 8.5), dpi=110)
    gs = fig.add_gridspec(4, 2, width_ratios=[2.2, 1.0],
                          height_ratios=[1.4, 1.4, 1.0, 1.0],
                          wspace=0.05, hspace=0.30)
    ax_scene = fig.add_subplot(gs[:, 0])
    ax_speed = fig.add_subplot(gs[0, 1])
    ax_rel   = fig.add_subplot(gs[1, 1])
    ax_long  = fig.add_subplot(gs[2, 1])
    ax_lat   = fig.add_subplot(gs[3, 1])

    imgs = []
    for f in frames:
        _draw_scene_frame(ax_scene, scenario, f, str(center_objects_id), view_radius=view_radius)
        ax_scene.set_title(f'{title}  t={f * DT:.1f}s', fontsize=9)
        _draw_speedometer         (ax_speed, speed_per_frame_disp[f] if f < T else np.nan)
        _draw_relative_speedometer(ax_rel,   rel_per_frame[f]        if f < T else np.nan)
        _draw_accel_bar(
            ax_long, a_long_per_frame[f] if f < T else np.nan,
            label='long accel', amax_g=A_LONG_MAX_G,
            warn_g=ACC_LONG_WARN_G, limit_g=ACC_LONG_LIMIT_G,
            neg_text='brake', pos_text='accel',
        )
        _draw_accel_bar(
            ax_lat, a_lat_per_frame[f] if f < T else np.nan,
            label='lat accel', amax_g=A_LAT_MAX_G,
            warn_g=ACC_LAT_WARN_G, limit_g=ACC_LAT_LIMIT_G,
            neg_text='right', pos_text='left',
        )
        buf = io.BytesIO()
        fig.savefig(buf, format='png', bbox_inches='tight', pad_inches=0.1)
        buf.seek(0)
        imgs.append(imageio.imread(buf))
    plt.close(fig)
    imageio.mimsave(output_path, imgs, fps=fps, loop=0)
    return output_path

## 4. Render GIFs

In [5]:
CFG = {
    'dataset':      DATASET,
    'split':        SPLIT,
    'view_radius':  70.0,
    'frame_stride': 2,
    'fps':          10,
    'max_frames':   40,
}

manifest: list[dict] = []
for label, top in picks.items():
    out_subdir = GIF_DIR / label
    out_subdir.mkdir(parents=True, exist_ok=True)
    for i, (_, row) in enumerate(top.iterrows()):
        gif_path = out_subdir / f'{label}_kdsc_{DATASET}_scenario={row["scenario_id"]}_center={row["center_objects_id"]}.gif'
        title = (f"KDSC {label} ({DATASET})  ext={row['extremeness']:.2f}  "
                 f"v_avg={row.get('avg_speed', float('nan')):.1f} "
                 f"|a|max={row.get('max_abs_accel', float('nan')):.1f}")
        try:
            render_agent_gif_with_dashboard(
                dataset=CFG['dataset'], split=CFG['split'],
                scenario_id=str(row['scenario_id']),
                center_objects_id=str(row['center_objects_id']),
                output_path=gif_path,
                title=title,
                view_radius=CFG['view_radius'],
                frame_stride=CFG['frame_stride'],
                fps=CFG['fps'],
                max_frames=CFG['max_frames'],
            )
            manifest.append({'label': label, 'rank': i, 'gif': str(gif_path),
                             'scenario_id': str(row['scenario_id']),
                             'center_objects_id': str(row['center_objects_id']),
                             'extremeness': float(row['extremeness'])})
            print('OK ', gif_path.name)
        except Exception as err:
            print('skip', row['scenario_id'], '->', err)

with open(GIF_DIR / 'manifest.json', 'w') as f:
    json.dump({'config': CFG, 'features': FEATURES, 'label_map': LABEL_MAP,
               'gifs': manifest}, f, indent=2)
print('\nWrote', len(manifest), 'gifs ->', GIF_DIR)

OK  normal_kdsc_av2_scenario=9f8e22b9-2eae-4de1-8816-3517b3101a71_center=43633.gif
OK  normal_kdsc_av2_scenario=3b9b45c5-023f-40e8-83de-0743cc8faee7_center=59219.gif
OK  normal_kdsc_av2_scenario=a912275c-830e-4e0f-b7bf-ee461d60d269_center=154405.gif
OK  normal_kdsc_av2_scenario=d974b9f7-9ed2-4671-adcd-0c54a711665b_center=36162.gif
OK  normal_kdsc_av2_scenario=06f616af-3a2e-4607-8890-20b3cc976e6f_center=51659.gif
OK  aggressive_kdsc_av2_scenario=c7dc1311-62ca-4c11-b5de-20d6b451bfe3_center=509.gif
OK  aggressive_kdsc_av2_scenario=03bfd760-d8d9-46f3-aa16-885efb85cf16_center=89470.gif
OK  aggressive_kdsc_av2_scenario=acc7df2d-82d5-4e8b-be90-fcfadab2b44f_center=59580.gif
OK  aggressive_kdsc_av2_scenario=518fd970-ee38-4bd7-a552-6c48558365cd_center=62225.gif
OK  aggressive_kdsc_av2_scenario=3200056a-a13d-4e10-81e2-34f2dc9a6afd_center=8948.gif

Wrote 10 gifs -> /fs/nexus-projects/pc_driving/yaghoubi/tail-risk-motion-prediction/artifacts/kdsc_replication/av2/figures/gifs


## 5. Inline preview

In [ ]:
from IPython.display import Image, display
for entry in manifest:
    print(f"[{entry['label']}] {entry['gif']}")
    try:
        display(Image(filename=entry['gif']))
    except Exception as err:
        print('  preview failed:', err)